# Самостоятельная работа. Анализ и обработка больших данных с Dask

**Цель работы:** изучить возможности библиотеки Dask для построения ETL-пайплайнов и визуализации графов вычислений (DAG) при работе с большими объемами данных.

---

### Варианты заданий
| Варианты | Имя файла (Датасет для ETL) |
| :--- | :--- |
| 1, 7, 13, 19, 25 | Parking_Violations_Issued_-_Fiscal_Year_2014__August_2013___June_2014_.csv |
| 2, 8, 14, 20 | Parking_Violations_Issued_-_Fiscal_Year_2015.csv |
| 3, 9, 15, 21 | Parking_Violations_Issued_-_Fiscal_Year_2016.csv |
| 4, 10, 16, 22 | Parking_Violations_Issued_-_Fiscal_Year_2017.csv |
| 5, 11, 17, 23 | UK Property Price official data 1995-202304.zip |
| 6, 12, 18, 24 | Austin, TX House Listings.zip |

**Ваш вариант: 14**

In [ ]:
# Установка Dask с полным набором зависимостей и графов
# !pip install "dask[complete]" graphviz

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar
import sys
import os
import pandas as pd
from IPython.display import Image

# Инициализация клиента Dask (Оптимизированные настройки без жесткого лимита памяти)
client = Client(n_workers=2, threads_per_worker=2, processes=True)
client

### Шаг 1. Extract (Извлечение данных)
Ленивая загрузка датасета с оптимизацией типов данных.

In [ ]:
# Определение типов данных для больших и неоднородных файлов NYC Parking
dtypes = {
    'Issuer Command': 'object',
    'Issuer Squad': 'object',
    'House Number': 'object',
    'Time First Observed': 'object',
    'Violation Description': 'object',
    'Violation Legal Code': 'object',
    'Violation Post Code': 'object',
    'Unregistered Vehicle?': 'float64',
    'Violation Location': 'float64',
    'Date First Observed': 'object',
    'Feet From Curb': 'float64',
    'Law Section': 'object',
    'Vehicle Year': 'float64',
    'Meter Number': 'object',
    'Violation County': 'object',
    'Double Parking Violation': 'object',
    'Hydrant Violation': 'object',
    'No Standing or Stopping Violation': 'object',
    'Sub Division': 'object',
    'Vehicle Color': 'object',
    'Vehicle Body Type': 'object',
    'Vehicle Make': 'object',
    'Violation Time': 'object'
}

df = dd.read_csv('Parking_Violations_Issued_-_Fiscal_Year_2015.csv', dtype=dtypes)
df

### Шаг 2. Transform (Трансформация и очистка данных)
Анализ пропусков, удаление разреженных столбцов (>55%) и техническая очистка.

In [ ]:
# Подсчет пропущенных значений (построение графа вычислений)
missing_values = df.isnull().sum()

# Вычисление процента пропусков
mysize = df.index.size
missing_count = ((missing_values / mysize) * 100)

# Запуск реальных вычислений только для агрегированной статистики
with ProgressBar():
    missing_count_percent = missing_count.compute()

print(missing_count_percent.sort_values(ascending=False).head(15))

# Формирование списка столбцов, где пропусков > 55%
columns_to_drop = list(missing_count_percent[missing_count_percent > 55].index)
print("\nУдаляемые столбцы (пропуски > 55%):", columns_to_drop)

# Ленивое удаление столбцов
df_dropped = df.drop(columns=columns_to_drop)

# Удаление дополнительных технических и избыточных столбцов
additional_columns = [
    'Street Code1', 'Street Code2', 'Street Code3',
    'Issuer Code', 'Feet From Curb', 'Violation Post Code'
]

existing_extra = [c for c in additional_columns if c in df_dropped.columns]
df_final = df_dropped.drop(columns=existing_extra)

# Преобразование формата даты
df_final['Issue Date'] = dd.to_datetime(df_final['Issue Date'], errors='coerce')

df_final.head()

### Шаг 3. Load (Сохранение результатов)
Сохранение очищенного датасета в единый CSV файл.

In [ ]:
df_final.to_csv('55_cleaned_violations_2015.csv', 
                  single_file=True, 
                  index=False)

### Шаг 4. Визуализация направленных ациклических графов (DAG)

#### 4.1. Простой граф
Расчет количества нарушений, уникальных марок ТС и среднего на марку.

In [2]:
from dask import delayed
from IPython.display import Image

def get_total_violations():
    return len(df_final)

def get_unique_makes():
    return df_final['Vehicle Make'].nunique().compute()

def avg_violations_per_make(total, unique):
    if unique == 0: return 0
    return round(total / unique, 2)

x = delayed(get_total_violations)()
y = delayed(get_unique_makes)()
z = delayed(avg_violations_per_make)(x, y)

try:
    z.visualize(filename='simple_violation_analysis.png')
    display(Image('simple_violation_analysis.png'))
except:
    print("Graphviz not found.")

print("Результат вычисления DAG:", z.compute())

Graphviz not found.


NameError: name 'df_final' is not defined

#### 4.2. Сложный граф (Аналитика по районам)
Сравнение доли нарушений в утренние часы пик (8-10 утра) в разных районах Нью-Йорка.

In [ ]:
from dask import delayed
from IPython.display import Image

# Список районов для анализа
districts = ['NY', 'K', 'Q', 'BX', 'R']

def load_district_data(district):
    return df_final[df_final['Violation County'] == district]

def count_violations(district_data):
    return len(district_data)

def count_peak_hours(district_data):
    if district_data is None or len(district_data) == 0: return 0
    # Выделяем часы из Violation Time (первые 2 символа)
    hours = district_data['Violation Time'].astype(str).str[:2]
    peak = hours[hours.isin(['08', '09', '10'])]
    return len(peak)

def calculate_peak_percentage(total, peak):
    if total == 0: return 0
    return round((peak / total) * 100, 2)

layer1 = [delayed(load_district_data)(d) for d in districts]
layer2 = [delayed(count_violations)(d) for d in layer1]
layer3 = [delayed(count_peak_hours)(d) for d in layer1]
layer4 = [delayed(calculate_peak_percentage)(t, p) for t, p in zip(layer2, layer3)]

results = delayed(list)(layer4)

try:
    results.visualize(filename='complex_district_analysis.png')
    display(Image('complex_district_analysis.png'))
except:
    print("Graphviz error.")

print("Результаты по районам % (NY, K, Q, BX, R):", results.compute())

### Шаг 5. Контрольные вопросы

**1. В чем главное отличие архитектуры dask.dataframe от классического pandas.dataframe в контексте обработки Big Data?**

`pandas` работает в одном потоке и требует, чтобы весь датасет помещался в оперативную память. `dask.dataframe` — это виртуальный массив, состоящий из множества маленьких pandas DataFrames (партиций), распределенных по ядрам процессора. Это позволяет обрабатывать данные, размер которых на порядки больше доступной RAM.

**2. Что такое "ленивые вычисления" (lazy evaluation) и почему вызов метода .compute() следует откладывать на самый конец ETL-пайплайна?**

Ленивые вычисления означают построение логического плана (DAG) вместо мгновенного исполнения. Вызов `.compute()` запускает реальные вычисления. Откладывание этого вызова позволяет Dask оптимизировать граф: выполнять фильтрацию до загрузки лишних данных и объединять операции.

**3. Что такое DAG (Directed Acyclic Graph), как он формируется в Dask и какую роль играет в оптимизации планировщика задач?**

DAG — это граф, где узлы — это функции, а ребра — зависимости данных. В Dask он формируется автоматически при вызове задержанных методов. Он позволяет планировщику видеть зависимости, что дает возможность параллельного исполнения независимых веток и минимизации накладных расходов.
